# Bayesian distributions

In `mixle.stats`, every leaf can carry a conjugate prior via `prior=`, so fitting produces a posterior rather than a point estimate. This is the syntax tutorial; for the conjugacy and Dirichlet-process theory see the data-science notebook conjugate_and_nonparametric_bayes.


In [1]:
import numpy as np
from mixle.stats import (GaussianDistribution, CategoricalDistribution, PoissonDistribution,
                        CompositeDistribution, SequenceDistribution, IntegerCategoricalDistribution,
                        OptionalDistribution, GaussianEstimator, CategoricalEstimator,
                        PoissonEstimator, CompositeEstimator, SequenceEstimator, OptionalEstimator,
                        IntegerCategoricalEstimator, NormalGammaDistribution, GammaDistribution,
                        DictDirichletDistribution, DirichletProcessMixtureEstimator)
from mixle.inference.estimation import fit, optimize

## A conjugate leaf: prior in, posterior out

Give an estimator a prior; `fit` returns the distribution whose parameters are the posterior (MAP) estimate. A NormalGamma prior on a Gaussian acts as a regularizer that matters most when data is scarce.


In [2]:
rng = np.random.RandomState(1)
data = rng.randn(5) * 2.0 + 3.0                       # five points from N(3, 4)
prior = NormalGammaDistribution(0.0, 1.0, 2.0, 2.0)   # weakly informative, centered at 0
d = fit(data, GaussianEstimator(prior=prior))
print('n=5: sample mean %.3f -> MAP mu %.3f, sigma2 %.3f' % (data.mean(), d.mu, d.sigma2))

Iteration 1. OBJ=-13.767651, dOBJ=3.770679e-01, LL=-10.811391, MLL=-2.956260, VLL=-10.811391
Terminating 2. OBJ=-13.767651, dOBJ=0.000000e+00, LL=-10.811391, MLL=-2.956260, VLL=-10.811391
n=5: sample mean 3.111 -> MAP mu 3.124, sigma2 5.881


Categorical with a Dirichlet prior - the pseudo-counts smooth unseen categories:


In [3]:
obs = ['a'] * 60 + ['b'] * 40                          # 'c' never observed
d = fit(obs, CategoricalEstimator(prior=DictDirichletDistribution({'a': 2.0, 'b': 2.0, 'c': 2.0})))
print('P(a)=%.3f  P(b)=%.3f  P(c)=%.3f (smoothed, never seen)'
      % tuple(np.exp(d.log_density(v)) for v in 'abc'))

Iteration 1. OBJ=-69.569834, dOBJ=2.582822e+01, LL=-68.277584, MLL=-1.292249, VLL=-68.277584
Terminating 2. OBJ=-69.569834, dOBJ=0.000000e+00, LL=-68.277584, MLL=-1.292249, VLL=-68.277584
P(a)=0.592  P(b)=0.398  P(c)=0.010 (smoothed, never seen)


Other conjugate pairs work the same way (Poisson/Gamma shown):


In [4]:
counts = PoissonDistribution(6.0).sampler(seed=13).sample(200)
d_pois = fit(counts, PoissonEstimator(prior=GammaDistribution(1.0, 1.0)))
print('Poisson/Gamma: true lambda 6.0, MAP %.3f' % d_pois.lam)

Iteration 1. OBJ=-446.467905, dOBJ=3.583798e-01, LL=-440.706711, MLL=-5.761194, VLL=-440.706711
Terminating 2. OBJ=-446.467905, dOBJ=0.000000e+00, LL=-440.706711, MLL=-5.761194, VLL=-440.706711
Poisson/Gamma: true lambda 6.0, MAP 5.761


## More conjugate pairs, and the posterior protocol

The same `Estimator(prior=...)` + `fit` pattern covers every conjugate family. After fitting, two things are available: the point estimate in the model's own parameterization (`d.p`, `d.lam`, ...), which is the posterior mode (MAP), and the full posterior via `d.get_prior()` - the updated conjugate distribution itself. `d.expected_log_density(x)` scores `x` under the posterior (the variational E-step quantity), integrating over parameter uncertainty rather than plugging in a point.


In [5]:
import io
from mixle.stats import (BernoulliEstimator, BetaDistribution, ExponentialDistribution,
                        ExponentialEstimator, GeometricDistribution, GeometricEstimator)
_q = dict(out=io.StringIO())   # quiet the EM iteration log

bb = fit(list((np.random.RandomState(0).rand(200) < 0.3).astype(float)),
         BernoulliEstimator(prior=BetaDistribution(1.0, 1.0)), **_q)                # Beta-Bernoulli
eg = fit(list(ExponentialDistribution(2.0).sampler(seed=1).sample(200)),
         ExponentialEstimator(prior=GammaDistribution(2.0, 1.0)), **_q)             # Exponential-Gamma  (beta = scale/mean)
gb = fit(list(GeometricDistribution(0.3).sampler(seed=1).sample(200)),
         GeometricEstimator(prior=BetaDistribution(1.0, 1.0)), **_q)               # Geometric-Beta
print('Beta-Bernoulli    : p = %.3f         (true 0.3)' % bb.p)
print('Exponential-Gamma : scale = %.3f     (true 2.0)' % eg.beta)
print('Geometric-Beta    : p = %.3f         (true 0.3)' % gb.p)

# point estimate (MAP) vs the full posterior the fit carries
d = fit(list(PoissonDistribution(4.0).sampler(seed=1).sample(200)),
        PoissonEstimator(prior=GammaDistribution(2.0, 1.0)), **_q)
shape, scale = (float(v) for v in d.get_prior().get_parameters())
print('\nPoisson MAP rate (point) :', round(d.lam, 3))
print('Gamma posterior          : shape=%.1f scale=%.4f -> mean=%.3f' % (shape, scale, shape * scale))
print('E_post[log p(x=3)]       :', round(d.expected_log_density(3), 3), '(integrates over rate uncertainty)')

Beta-Bernoulli    : p = 0.305         (true 0.3)
Exponential-Gamma : scale = 1.958     (true 2.0)
Geometric-Beta    : p = 0.303         (true 0.3)

Poisson MAP rate (point) : 3.97
Gamma posterior          : shape=799.0 scale=0.0050 -> mean=3.975
E_post[log p(x=3)]       : -1.629 (integrates over rate uncertainty)


## Priors nest through composites

A record of leaves-with-priors fits as one object; each leaf updates its own posterior in a single pass.


In [6]:
truth = CompositeDistribution((GaussianDistribution(2.0, 1.0),
                              CategoricalDistribution({'x': 0.7, 'y': 0.3})))
rec_data = truth.sampler(seed=7).sample(300)
est = CompositeEstimator((GaussianEstimator(), CategoricalEstimator()))
d = fit(rec_data, est)
print('fitted record fields:', round(d.dists[0].mu, 2), '|', {k: round(np.exp(d.dists[1].log_density(k)), 2) for k in 'xy'})

Iteration 1. OBJ=-595.894417, dOBJ=1.060224e+01, LL=-595.894417, MLL=0.000000, VLL=-595.894417
Terminating 2. OBJ=-595.894417, dOBJ=0.000000e+00, LL=-595.894417, MLL=0.000000, VLL=-595.894417
fitted record fields: 2.12 | {'x': np.float64(0.66), 'y': np.float64(0.34)}


## Dirichlet process mixture: inferring the component count

A `DirichletProcessMixtureEstimator` fits a mixture whose number of active components is inferred from the data - no `K` to set.


In [7]:
comps = [CompositeDistribution((GaussianDistribution(m, 1.0), CategoricalDistribution(p)))
         for m, p in [(-6.0, {'x': 0.8, 'y': 0.2}), (0.0, {'x': 0.5, 'y': 0.5}), (6.0, {'x': 0.1, 'y': 0.9})]]
mix_data = []
for c in comps:
    mix_data += list(c.sampler(seed=3).sample(120))
comp_est = CompositeEstimator((GaussianEstimator(prior=NormalGammaDistribution(0.0, 1.0, 2.0, 2.0)),
                               CategoricalEstimator(prior=DictDirichletDistribution({'x': 1.0, 'y': 1.0}))))
dpm = optimize(mix_data, DirichletProcessMixtureEstimator([comp_est] * 12),
               max_its=150, rng=np.random.RandomState(5))
active = int((np.asarray(dpm.w) > 0.01).sum())
print('fit a DP mixture (max 12 sticks); active components:', active)

Iteration 1: ELBO=-1.431052e+03, dELBO=3.262379e+02
Iteration 2: ELBO=-1.404431e+03, dELBO=2.662119e+01
Iteration 3: ELBO=-1.383572e+03, dELBO=2.085889e+01
Iteration 4: ELBO=-1.368420e+03, dELBO=1.515232e+01
Iteration 5: ELBO=-1.350664e+03, dELBO=1.775575e+01
Iteration 6: ELBO=-1.326790e+03, dELBO=2.387384e+01
Iteration 7: ELBO=-1.308406e+03, dELBO=1.838491e+01
Iteration 8: ELBO=-1.294817e+03, dELBO=1.358907e+01
Iteration 9: ELBO=-1.278409e+03, dELBO=1.640708e+01
Iteration 10: ELBO=-1.256927e+03, dELBO=2.148203e+01
Iteration 11: ELBO=-1.234747e+03, dELBO=2.218088e+01
Iteration 12: ELBO=-1.220935e+03, dELBO=1.381181e+01
Iteration 13: ELBO=-1.212090e+03, dELBO=8.844233e+00
Iteration 14: ELBO=-1.204809e+03, dELBO=7.281359e+00
Iteration 15: ELBO=-1.197785e+03, dELBO=7.024039e+00
Iteration 16: ELBO=-1.192152e+03, dELBO=5.632790e+00
Iteration 17: ELBO=-1.187150e+03, dELBO=5.002793e+00
Iteration 18: ELBO=-1.182198e+03, dELBO=4.951176e+00
Iteration 19: ELBO=-1.178395e+03, dELBO=3.802911e+00
It